In [5]:
# ============================================================
# NOTEBOOK 05 — PRODUCT & CATEGORY ANALYSIS
# Cell 1: Load Raw Datasets
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

orders = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

products = pd.read_csv("../data/raw/olist_products_dataset.csv")

category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

print("Datasets loaded successfully.\n")

# ------------------------------------------------------------
# Display shapes
# ------------------------------------------------------------

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Datasets loaded successfully.

customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
category_translation: (71, 2)


In [6]:
# ============================================================
# NOTEBOOK 05 — PRODUCT & CATEGORY ANALYSIS
# Cell 2: Data Validation
# ============================================================

# ------------------------------------------------------------
# Check important columns
# ------------------------------------------------------------

print("Orders columns:")
print(orders.columns.tolist())

print("\nOrder items columns:")
print(order_items.columns.tolist())

print("\nProducts columns:")
print(products.columns.tolist())

print("\nCategory translation columns:")
print(category_translation.columns.tolist())


# ------------------------------------------------------------
# Missing values in important datasets
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

print("\nOrders:")
print(orders.isna().sum())

print("\nOrder items:")
print(order_items.isna().sum())

print("\nProducts:")
print(products.isna().sum())


# ------------------------------------------------------------
# Duplicate checks
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DUPLICATE CHECKS")
print("=" * 60)

print("\nDuplicate order IDs:", orders["order_id"].duplicated().sum())

print(
    "Duplicate order-item combinations:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "Duplicate product IDs:",
    products["product_id"].duplicated().sum()
)

print(
    "Duplicate category translations:",
    category_translation[
        "product_category_name"
    ].duplicated().sum()
)


# ------------------------------------------------------------
# Unique counts
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("UNIQUE COUNTS")
print("=" * 60)

print("Unique orders:", orders["order_id"].nunique())
print("Unique customers:", customers["customer_unique_id"].nunique())
print("Unique products:", products["product_id"].nunique())
print(
    "Unique product categories:",
    products["product_category_name"].nunique()
)

Orders columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Order items columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Products columns:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Category translation columns:
['product_category_name', 'product_category_name_english']

MISSING VALUES

Orders:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order items:
ord

In [7]:
# ============================================================
# NOTEBOOK 05 — PRODUCT & CATEGORY ANALYSIS
# Cell 3: Build Product-Level Sales Dataset
# ============================================================

# ------------------------------------------------------------
# 1. Merge order items with product information
# ------------------------------------------------------------

product_sales = order_items.merge(
    products[
        [
            "product_id",
            "product_category_name"
        ]
    ],
    on="product_id",
    how="left"
)


# ------------------------------------------------------------
# 2. Translate product category names into English
# ------------------------------------------------------------

product_sales = product_sales.merge(
    category_translation,
    on="product_category_name",
    how="left"
)


# ------------------------------------------------------------
# 3. Handle missing category names
# ------------------------------------------------------------

product_sales["product_category_name_english"] = (
    product_sales["product_category_name_english"]
    .fillna("Unknown")
)


# ------------------------------------------------------------
# 4. Calculate item-level revenue
# ------------------------------------------------------------

product_sales["item_revenue"] = (
    product_sales["price"] +
    product_sales["freight_value"]
)


# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

print("Product sales dataset shape:", product_sales.shape)

print("\nColumns:")
print(product_sales.columns.tolist())

print("\nMissing values:")
print(
    product_sales[
        [
            "product_id",
            "product_category_name_english",
            "price",
            "freight_value",
            "item_revenue"
        ]
    ].isna().sum()
)

print("\nSample:")
display(product_sales.head())

print("\nTotal item revenue:")
print(
    f"${product_sales['item_revenue'].sum():,.2f}"
)

print("\nUnique categories:")
print(
    product_sales[
        "product_category_name_english"
    ].nunique()
)

Product sales dataset shape: (112650, 10)

Columns:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_category_name_english', 'item_revenue']

Missing values:
product_id                       0
product_category_name_english    0
price                            0
freight_value                    0
item_revenue                     0
dtype: int64

Sample:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_category_name_english,item_revenue
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,cool_stuff,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,pet_shop,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,furniture_decor,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,perfumery,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,garden_tools,218.04



Total item revenue:
$15,843,553.24

Unique categories:
72


In [8]:
# ============================================================
# NOTEBOOK 05 — PRODUCT & CATEGORY ANALYSIS
# Cell 4: Category-Level Sales Performance
# ============================================================

category_sales = (
    product_sales
    .groupby("product_category_name_english")
    .agg(
        items_sold=("order_item_id", "count"),
        unique_orders=("order_id", "nunique"),
        unique_products=("product_id", "nunique"),
        total_revenue=("item_revenue", "sum"),
        average_item_revenue=("item_revenue", "mean"),
        median_item_revenue=("item_revenue", "median")
    )
    .reset_index()
)


# ------------------------------------------------------------
# Calculate revenue share
# ------------------------------------------------------------

total_category_revenue = category_sales["total_revenue"].sum()

category_sales["revenue_share_pct"] = (
    category_sales["total_revenue"]
    / total_category_revenue
    * 100
)


# ------------------------------------------------------------
# Sort by total revenue
# ------------------------------------------------------------

category_sales = category_sales.sort_values(
    "total_revenue",
    ascending=False
).reset_index(drop=True)


# ------------------------------------------------------------
# Display top 20 categories
# ------------------------------------------------------------

print("Top 20 Product Categories by Revenue:")
print("=" * 80)

display(
    category_sales.head(20).round(2)
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nTotal categories:", len(category_sales))

print(
    f"Total category revenue: "
    f"${category_sales['total_revenue'].sum():,.2f}"
)

print(
    f"Top 10 categories revenue share: "
    f"{category_sales.head(10)['revenue_share_pct'].sum():.2f}%"
)

Top 20 Product Categories by Revenue:


,product_category_name_english,items_sold,unique_orders,unique_products,total_revenue,average_item_revenue,median_item_revenue,revenue_share_pct
0,health_beauty,9670,8836,2444,1441248.07,149.04,96.11,9.10
1,watches_gifts,5991,5624,1329,1305541.61,217.92,146.02,8.24
2,bed_bath_table,11115,9417,3029,1241681.72,111.71,94.67,7.84
3,sports_leisure,8641,7720,2867,1156656.48,133.86,96.47,7.30
4,computers_accessories,7827,6689,1639,1059272.40,135.34,97.13,6.69
5,furniture_decor,8334,6449,2657,902511.79,108.29,81.24,5.70
6,housewares,6964,5884,2335,778397.77,111.77,76.02,4.91
7,cool_stuff,3796,3632,789,719329.95,189.50,145.51,4.54
8,auto,4235,3897,1900,685384.32,161.84,104.80,4.33
9,garden_tools,4347,3518,753,584219.21,134.40,77.57,3.69



Total categories: 72
Total category revenue: $15,843,553.24
Top 10 categories revenue share: 62.32%


In [9]:
# ============================================================
# NOTEBOOK 05 — PRODUCT & CATEGORY ANALYSIS
# Cell 5: Attach Customer Type to Product Purchases
# ============================================================

# ------------------------------------------------------------
# Build customer identity mapping
# ------------------------------------------------------------

customer_identity = (
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_state"
        ]
    ]
    .drop_duplicates("customer_id")
)


# ------------------------------------------------------------
# Determine customer order counts
# ------------------------------------------------------------

customer_order_counts = (
    orders
    .merge(
        customer_identity[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left"
    )
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique")
    )
    .reset_index()
)


# ------------------------------------------------------------
# Classify customers
# ------------------------------------------------------------

customer_order_counts["customer_type"] = np.where(
    customer_order_counts["total_orders"] > 1,
    "Repeat",
    "One-time"
)


# ------------------------------------------------------------
# Attach customer_unique_id to product sales
# ------------------------------------------------------------

product_sales_customer = (
    product_sales
    .merge(
        orders[
            [
                "order_id",
                "customer_id"
            ]
        ],
        on="order_id",
        how="left"
    )
    .merge(
        customer_identity,
        on="customer_id",
        how="left"
    )
    .merge(
        customer_order_counts[
            [
                "customer_unique_id",
                "customer_type"
            ]
        ],
        on="customer_unique_id",
        how="left"
    )
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print(
    "Product sales with customer information:",
    product_sales_customer.shape
)

print(
    "\nMissing customer_unique_id:",
    product_sales_customer["customer_unique_id"].isna().sum()
)

print(
    "Missing customer_type:",
    product_sales_customer["customer_type"].isna().sum()
)

print("\nCustomer type distribution within product purchases:")
print(
    product_sales_customer["customer_type"]
    .value_counts()
)

print("\nSample:")
display(product_sales_customer.head())

Product sales with customer information: (112650, 14)

Missing customer_unique_id: 0
Missing customer_type: 0

Customer type distribution within product purchases:
customer_type
One-time    105082
Repeat        7568
Name: count, dtype: int64

Sample:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_category_name_english,item_revenue,customer_id,customer_unique_id,customer_state,customer_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,cool_stuff,72.19,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,RJ,One-time
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,pet_shop,259.83,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,SP,Repeat
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,furniture_decor,216.87,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,MG,One-time
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,perfumery,25.78,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,SP,One-time
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,garden_tools,218.04,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,SP,One-time


In [10]:
# ============================================================
# NOTEBOOK 05 — PRODUCT & CATEGORY ANALYSIS
# Cell 6: Category Repeat-Customer Analysis
# ============================================================

category_repeat_analysis = (
    product_sales_customer
    .groupby("product_category_name_english")
    .agg(
        total_items=("order_item_id", "count"),
        one_time_items=(
            "customer_type",
            lambda x: (x == "One-time").sum()
        ),
        repeat_items=(
            "customer_type",
            lambda x: (x == "Repeat").sum()
        ),
        total_revenue=("item_revenue", "sum"),
        repeat_revenue=(
            "item_revenue",
            lambda x: x[
                product_sales_customer.loc[
                    x.index,
                    "customer_type"
                ] == "Repeat"
            ].sum()
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Repeat purchase share
# ------------------------------------------------------------

category_repeat_analysis["repeat_item_share_pct"] = (
    category_repeat_analysis["repeat_items"]
    / category_repeat_analysis["total_items"]
    * 100
)


# ------------------------------------------------------------
# Repeat revenue share
# ------------------------------------------------------------

category_repeat_analysis["repeat_revenue_share_pct"] = (
    category_repeat_analysis["repeat_revenue"]
    / category_repeat_analysis["total_revenue"]
    * 100
)


# ------------------------------------------------------------
# Average revenue per item
# ------------------------------------------------------------

category_repeat_analysis["average_item_revenue"] = (
    category_repeat_analysis["total_revenue"]
    / category_repeat_analysis["total_items"]
)


# ------------------------------------------------------------
# Sort by repeat-item share
# ------------------------------------------------------------

category_repeat_analysis = (
    category_repeat_analysis
    .sort_values(
        "repeat_item_share_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("Categories ranked by repeat-customer purchase share:")
print("=" * 90)

display(
    category_repeat_analysis.round(2)
)


# ------------------------------------------------------------
# Overall comparison
# ------------------------------------------------------------

print("\nOverall repeat-item share:")

overall_repeat_share = (
    product_sales_customer["customer_type"]
    .eq("Repeat")
    .mean()
    * 100
)

print(f"{overall_repeat_share:.2f}%")

Categories ranked by repeat-customer purchase share:


,product_category_name_english,total_items,one_time_items,repeat_items,total_revenue,repeat_revenue,repeat_item_share_pct,repeat_revenue_share_pct,average_item_revenue
0,diapers_and_hygiene,39,29,10,2141.27,382.00,25.64,17.84,54.90
1,arts_and_craftmanship,24,19,5,2184.14,428.12,20.83,19.60,91.01
2,home_appliances,771,633,138,94990.43,11325.20,17.90,11.92,123.20
3,fashio_female_clothing,48,40,8,3425.39,414.80,16.67,12.11,71.36
4,la_cuisine,14,12,2,2388.54,351.79,14.29,14.73,170.61
...,...,...,...,...,...,...,...,...,...
67,books_technical,267,261,6,23379.12,330.69,2.25,1.41,87.56
68,small_appliances_home_oven_and_coffee,76,75,1,50193.57,793.09,1.32,1.58,660.44
69,cds_dvds_musicals,14,14,0,954.99,0.00,0.00,0.00,68.21
70,flowers,33,33,0,1598.91,0.00,0.00,0.00,48.45



Overall repeat-item share:
6.72%


In [11]:
# ============================================================
# NOTEBOOK 05 — PRODUCT & CATEGORY ANALYSIS
# Cell 7: High-Volume / High-Repeat Categories
# ============================================================

# ------------------------------------------------------------
# Define minimum volume threshold
# ------------------------------------------------------------

minimum_items = 500


high_volume_categories = (
    category_repeat_analysis[
        category_repeat_analysis["total_items"] >= minimum_items
    ]
    .copy()
)


# ------------------------------------------------------------
# Rank by repeat-item share
# ------------------------------------------------------------

high_volume_categories = (
    high_volume_categories
    .sort_values(
        "repeat_item_share_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    f"Categories with at least {minimum_items} items:"
)

display(
    high_volume_categories[
        [
            "product_category_name_english",
            "total_items",
            "repeat_items",
            "total_revenue",
            "repeat_revenue",
            "repeat_item_share_pct",
            "repeat_revenue_share_pct",
            "average_item_revenue"
        ]
    ].round(2)
)


# ------------------------------------------------------------
# Compare against overall repeat-item baseline
# ------------------------------------------------------------

print(
    f"\nOverall repeat-item baseline: "
    f"{overall_repeat_share:.2f}%"
)


high_volume_categories[
    "repeat_share_lift_pct"
] = (
    high_volume_categories["repeat_item_share_pct"]
    - overall_repeat_share
)


print("\nCategories above the overall repeat baseline:")

display(
    high_volume_categories[
        high_volume_categories["repeat_item_share_pct"]
        > overall_repeat_share
    ][
        [
            "product_category_name_english",
            "total_items",
            "repeat_items",
            "repeat_item_share_pct",
            "repeat_share_lift_pct",
            "total_revenue"
        ]
    ].round(2)
)

Categories with at least 500 items:


,product_category_name_english,total_items,repeat_items,total_revenue,repeat_revenue,repeat_item_share_pct,repeat_revenue_share_pct,average_item_revenue
0,home_appliances,771,138,94990.43,11325.20,17.90,11.92,123.20
1,fashion_bags_accessories,2031,263,184273.54,20959.77,12.95,11.37,90.73
2,bed_bath_table,11115,1114,1241681.72,118948.57,10.02,9.58,111.71
3,furniture_decor,8334,821,902511.79,81372.77,9.85,9.02,108.29
4,furniture_living_room,503,48,86884.73,7589.87,9.54,8.74,172.73
5,food,510,44,36664.44,3406.61,8.63,9.29,71.89
6,sports_leisure,8641,687,1156656.48,89411.40,7.95,7.73,133.86
7,home_construction,604,45,96920.36,6141.21,7.45,6.34,160.46
8,computers_accessories,7827,552,1059272.40,73247.67,7.05,6.91,135.34
9,construction_tools_construction,929,61,165328.00,9244.63,6.57,5.59,177.96



Overall repeat-item baseline: 6.72%

Categories above the overall repeat baseline:


,product_category_name_english,total_items,repeat_items,repeat_item_share_pct,repeat_share_lift_pct,total_revenue
0,home_appliances,771,138,17.90,11.18,94990.43
1,fashion_bags_accessories,2031,263,12.95,6.23,184273.54
2,bed_bath_table,11115,1114,10.02,3.30,1241681.72
3,furniture_decor,8334,821,9.85,3.13,902511.79
4,furniture_living_room,503,48,9.54,2.82,86884.73
5,food,510,44,8.63,1.91,36664.44
6,sports_leisure,8641,687,7.95,1.23,1156656.48
7,home_construction,604,45,7.45,0.73,96920.36
8,computers_accessories,7827,552,7.05,0.33,1059272.40


In [12]:
# ============================================================
# NOTEBOOK 05 — PRODUCT & CATEGORY ANALYSIS
# Cell 8: One-Time vs Repeat Category Spending
# ============================================================

category_customer_type = (
    product_sales_customer
    .groupby(
        [
            "product_category_name_english",
            "customer_type"
        ]
    )
    .agg(
        items=("order_item_id", "count"),
        total_revenue=("item_revenue", "sum"),
        average_item_revenue=("item_revenue", "mean"),
        median_item_revenue=("item_revenue", "median")
    )
    .reset_index()
)


# ------------------------------------------------------------
# Pivot one-time vs repeat results
# ------------------------------------------------------------

category_spending_comparison = (
    category_customer_type
    .pivot(
        index="product_category_name_english",
        columns="customer_type",
        values=[
            "items",
            "total_revenue",
            "average_item_revenue",
            "median_item_revenue"
        ]
    )
)


# Flatten column names
category_spending_comparison.columns = [
    f"{metric}_{customer_type.lower().replace('-', '_')}"
    for metric, customer_type
    in category_spending_comparison.columns
]


category_spending_comparison = (
    category_spending_comparison
    .reset_index()
)


# ------------------------------------------------------------
# Calculate repeat vs one-time average item value difference
# ------------------------------------------------------------

category_spending_comparison["repeat_vs_onetime_avg_value_pct"] = (
    (
        category_spending_comparison[
            "average_item_revenue_repeat"
        ]
        /
        category_spending_comparison[
            "average_item_revenue_one_time"
        ]
    ) - 1
) * 100


# ------------------------------------------------------------
# Keep categories with meaningful volume
# ------------------------------------------------------------

category_spending_comparison = (
    category_spending_comparison
    .merge(
        category_repeat_analysis[
            [
                "product_category_name_english",
                "total_items"
            ]
        ],
        on="product_category_name_english",
        how="left"
    )
)


category_spending_comparison = (
    category_spending_comparison[
        category_spending_comparison["total_items"] >= 500
    ]
    .sort_values(
        "repeat_vs_onetime_avg_value_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    category_spending_comparison[
        [
            "product_category_name_english",
            "items_one_time",
            "items_repeat",
            "average_item_revenue_one_time",
            "average_item_revenue_repeat",
            "repeat_vs_onetime_avg_value_pct",
            "median_item_revenue_one_time",
            "median_item_revenue_repeat"
        ]
    ].round(2)
)

,product_category_name_english,items_one_time,items_repeat,average_item_revenue_one_time,average_item_revenue_repeat,repeat_vs_onetime_avg_value_pct,median_item_revenue_one_time,median_item_revenue_repeat
0,small_appliances,653.0,26.0,294.67,547.99,85.97,117.93,226.61
1,consoles_games,1097.0,40.0,154.14,204.91,32.93,75.51,92.00
2,food,466.0,44.0,71.37,77.42,8.48,64.39,57.67
3,cool_stuff,3656.0,140.0,189.05,201.05,6.34,145.29,147.40
4,perfumery,3231.0,188.0,132.38,136.31,2.97,99.33,82.76
5,computers_accessories,7275.0,552.0,135.54,132.70,-2.10,95.65,106.60
6,office_furniture,1610.0,81.0,202.82,197.52,-2.61,178.08,174.07
7,electronics,2676.0,91.0,74.82,72.58,-3.00,37.00,37.00
8,sports_leisure,7954.0,687.0,134.18,130.15,-3.00,96.22,104.27
9,luggage_accessories,1040.0,52.0,156.73,151.55,-3.30,124.43,132.68


In [14]:
# ============================================================
# STATISTICAL TEST:
# Repeat vs One-time Item Revenue by Product Category
# ============================================================

from scipy.stats import mannwhitneyu
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Prepare category-level data
# ------------------------------------------------------------

category_test_data = product_sales_customer[
    [
        "product_category_name_english",
        "item_revenue",
        "customer_type"
    ]
].copy()

# Remove missing values
category_test_data = category_test_data.dropna(
    subset=[
        "product_category_name_english",
        "item_revenue",
        "customer_type"
    ]
)

# ------------------------------------------------------------
# 2. Test each category
# ------------------------------------------------------------

results = []

for category, group in category_test_data.groupby(
    "product_category_name_english"
):

    one_time = group.loc[
        group["customer_type"] == "One-time",
        "item_revenue"
    ]

    repeat = group.loc[
        group["customer_type"] == "Repeat",
        "item_revenue"
    ]

    # Require enough observations in both groups
    if len(one_time) < 10 or len(repeat) < 10:
        continue

    # Mann-Whitney U test
    statistic, p_value = mannwhitneyu(
        repeat,
        one_time,
        alternative="two-sided"
    )

    # Percentage difference in means
    one_time_mean = one_time.mean()
    repeat_mean = repeat.mean()

    value_difference_pct = (
        (repeat_mean - one_time_mean)
        / one_time_mean
    ) * 100

    results.append({
        "product_category_name_english": category,
        "one_time_items": len(one_time),
        "repeat_items": len(repeat),
        "one_time_avg_revenue": one_time_mean,
        "repeat_avg_revenue": repeat_mean,
        "repeat_vs_onetime_pct": value_difference_pct,
        "u_statistic": statistic,
        "p_value": p_value
    })

# ------------------------------------------------------------
# 3. Create results dataframe
# ------------------------------------------------------------

category_significance = pd.DataFrame(results)

# ------------------------------------------------------------
# 4. Multiple-testing correction
# ------------------------------------------------------------
# We are testing many categories, so control the False
# Discovery Rate using Benjamini-Hochberg.

# ============================================================
# BENJAMINI-HOCHBERG FDR CORRECTION
# ============================================================

if len(category_significance) > 0:

    p_values = category_significance["p_value"].to_numpy()

    n = len(p_values)

    # Sort p-values
    sorted_indices = np.argsort(p_values)
    sorted_p_values = p_values[sorted_indices]

    # Benjamini-Hochberg adjusted p-values
    adjusted_sorted = (
        sorted_p_values * n /
        np.arange(1, n + 1)
    )

    # Ensure adjusted p-values are monotonic
    adjusted_sorted = np.minimum.accumulate(
        adjusted_sorted[::-1]
    )[::-1]

    # Cap at 1
    adjusted_sorted = np.minimum(
        adjusted_sorted,
        1.0
    )

    # Put adjusted values back into original order
    adjusted_p_values = np.empty(n)
    adjusted_p_values[sorted_indices] = adjusted_sorted

    category_significance["adjusted_p_value"] = adjusted_p_values

    # Significance at alpha = 0.05
    category_significance["significant"] = (
        category_significance["adjusted_p_value"] < 0.05
    )

# Sort by adjusted p-value
category_significance = category_significance.sort_values(
    "adjusted_p_value"
).reset_index(drop=True)

print("Category-level statistical significance:")
print("=" * 80)

display(category_significance.round(4))

# ------------------------------------------------------------
# 5. Sort by statistical significance
# ------------------------------------------------------------

category_significance = category_significance.sort_values(
    "adjusted_p_value"
)

print("Category-level statistical significance:")
print("=" * 80)

display(category_significance.round(4))

Category-level statistical significance:


,product_category_name_english,one_time_items,repeat_items,one_time_avg_revenue,repeat_avg_revenue,repeat_vs_onetime_pct,u_statistic,p_value,adjusted_p_value,significant
0,computers_accessories,7275,552,135.5360,132.6951,-2.0961,2226683.0,0.0000,0.0003,True
1,health_beauty,9103,567,151.0052,117.5440,-22.1590,2301512.0,0.0000,0.0003,True
2,watches_gifts,5667,324,219.7382,186.0662,-15.3237,782348.5,0.0000,0.0003,True
3,home_appliances,633,138,132.1726,82.0667,-37.9095,34276.5,0.0001,0.0009,True
4,furniture_bedroom,94,15,244.7223,110.4740,-54.8574,290.0,0.0003,0.0026,True
5,bed_bath_table,10001,1114,112.2621,106.7761,-4.8868,5205509.0,0.0003,0.0027,True
6,fashion_bags_accessories,1768,263,92.3720,79.6949,-13.7240,201199.5,0.0004,0.0029,True
7,baby,2914,151,157.8518,133.3637,-15.5133,185716.5,0.0012,0.0069,True
8,diapers_and_hygiene,29,10,60.6645,38.2000,-37.0307,45.0,0.0013,0.0069,True
9,furniture_decor,7513,821,109.2958,99.1142,-9.3156,2884469.5,0.0023,0.0112,True


Category-level statistical significance:


,product_category_name_english,one_time_items,repeat_items,one_time_avg_revenue,repeat_avg_revenue,repeat_vs_onetime_pct,u_statistic,p_value,adjusted_p_value,significant
0,computers_accessories,7275,552,135.5360,132.6951,-2.0961,2226683.0,0.0000,0.0003,True
1,health_beauty,9103,567,151.0052,117.5440,-22.1590,2301512.0,0.0000,0.0003,True
2,watches_gifts,5667,324,219.7382,186.0662,-15.3237,782348.5,0.0000,0.0003,True
3,home_appliances,633,138,132.1726,82.0667,-37.9095,34276.5,0.0001,0.0009,True
4,furniture_bedroom,94,15,244.7223,110.4740,-54.8574,290.0,0.0003,0.0026,True
5,bed_bath_table,10001,1114,112.2621,106.7761,-4.8868,5205509.0,0.0003,0.0027,True
6,fashion_bags_accessories,1768,263,92.3720,79.6949,-13.7240,201199.5,0.0004,0.0029,True
7,baby,2914,151,157.8518,133.3637,-15.5133,185716.5,0.0012,0.0069,True
8,diapers_and_hygiene,29,10,60.6645,38.2000,-37.0307,45.0,0.0013,0.0069,True
9,furniture_decor,7513,821,109.2958,99.1142,-9.3156,2884469.5,0.0023,0.0112,True


In [15]:
# ============================================================
# CATEGORY RETENTION OPPORTUNITY ANALYSIS
# ============================================================

# Start from the category repeat analysis
category_opportunity = category_repeat_analysis.copy()

# Overall repeat-item baseline
overall_repeat_baseline = (
    category_opportunity["repeat_items"].sum()
    / category_opportunity["total_items"].sum()
) * 100

print(f"Overall repeat-item baseline: {overall_repeat_baseline:.2f}%")

# ------------------------------------------------------------
# 1. Calculate repeat-item share
# ------------------------------------------------------------

category_opportunity["repeat_item_share_pct"] = (
    category_opportunity["repeat_items"]
    / category_opportunity["total_items"]
) * 100

# ------------------------------------------------------------
# 2. Calculate lift over overall repeat baseline
# ------------------------------------------------------------

category_opportunity["repeat_share_lift_pct"] = (
    category_opportunity["repeat_item_share_pct"]
    - overall_repeat_baseline
)

# ------------------------------------------------------------
# 3. Add statistical significance results
# ------------------------------------------------------------

significance_columns = [
    "product_category_name_english",
    "repeat_vs_onetime_pct",
    "adjusted_p_value",
    "significant"
]

category_opportunity = category_opportunity.merge(
    category_significance[significance_columns],
    on="product_category_name_english",
    how="left"
)

# ------------------------------------------------------------
# 4. Create opportunity indicators
# ------------------------------------------------------------

category_opportunity["high_repeat_penetration"] = (
    category_opportunity["repeat_item_share_pct"]
    > overall_repeat_baseline
)

category_opportunity["statistically_significant"] = (
    category_opportunity["significant"]
    == True
)

# ------------------------------------------------------------
# 5. Revenue contribution
# ------------------------------------------------------------

total_revenue = category_opportunity["total_revenue"].sum()

category_opportunity["revenue_share_pct"] = (
    category_opportunity["total_revenue"]
    / total_revenue
) * 100

# ------------------------------------------------------------
# 6. Sort by repeat-item share
# ------------------------------------------------------------

category_opportunity = category_opportunity.sort_values(
    "repeat_item_share_pct",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

display(
    category_opportunity[
        [
            "product_category_name_english",
            "total_items",
            "repeat_items",
            "repeat_item_share_pct",
            "repeat_share_lift_pct",
            "total_revenue",
            "revenue_share_pct",
            "repeat_vs_onetime_pct",
            "adjusted_p_value",
            "significant"
        ]
    ].round(2)
)

Overall repeat-item baseline: 6.72%


,product_category_name_english,total_items,repeat_items,repeat_item_share_pct,repeat_share_lift_pct,total_revenue,revenue_share_pct,repeat_vs_onetime_pct,adjusted_p_value,significant
0,diapers_and_hygiene,39,10,25.64,18.92,2141.27,0.01,-37.03,0.01,True
1,arts_and_craftmanship,24,5,20.83,14.12,2184.14,0.01,NaN,NaN,NaN
2,home_appliances,771,138,17.90,11.18,94990.43,0.60,-37.91,0.00,True
3,fashio_female_clothing,48,8,16.67,9.95,3425.39,0.02,NaN,NaN,NaN
4,la_cuisine,14,2,14.29,7.57,2388.54,0.02,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
67,books_technical,267,6,2.25,-4.47,23379.12,0.15,NaN,NaN,NaN
68,small_appliances_home_oven_and_coffee,76,1,1.32,-5.40,50193.57,0.32,NaN,NaN,NaN
69,cds_dvds_musicals,14,0,0.00,-6.72,954.99,0.01,NaN,NaN,NaN
70,flowers,33,0,0.00,-6.72,1598.91,0.01,NaN,NaN,NaN


In [16]:
# ============================================================
# HIGH-OPPORTUNITY CATEGORIES
# ============================================================

high_opportunity_categories = category_opportunity[
    (category_opportunity["high_repeat_penetration"]) &
    (category_opportunity["total_revenue"] >= 50000)
].copy()

high_opportunity_categories = high_opportunity_categories.sort_values(
    [
        "repeat_item_share_pct",
        "total_revenue"
    ],
    ascending=[False, False]
)

print("High-opportunity categories:")
print("=" * 80)

display(
    high_opportunity_categories[
        [
            "product_category_name_english",
            "total_items",
            "repeat_items",
            "repeat_item_share_pct",
            "repeat_share_lift_pct",
            "total_revenue",
            "revenue_share_pct",
            "repeat_vs_onetime_pct",
            "adjusted_p_value",
            "significant"
        ]
    ].round(2)
)


High-opportunity categories:


,product_category_name_english,total_items,repeat_items,repeat_item_share_pct,repeat_share_lift_pct,total_revenue,revenue_share_pct,repeat_vs_onetime_pct,adjusted_p_value,significant
2,home_appliances,771,138,17.90,11.18,94990.43,0.60,-37.91,0.00,True
6,fashion_bags_accessories,2031,263,12.95,6.23,184273.54,1.16,-13.72,0.00,True
12,bed_bath_table,11115,1114,10.02,3.30,1241681.72,7.84,-4.89,0.00,True
14,furniture_decor,8334,821,9.85,3.13,902511.79,5.70,-9.32,0.01,True
15,furniture_living_room,503,48,9.54,2.82,86884.73,0.55,-9.27,0.15,False
19,sports_leisure,8641,687,7.95,1.23,1156656.48,7.30,-3.00,0.57,False
20,air_conditioning,297,23,7.74,1.03,61774.19,0.39,-11.20,0.97,False
21,home_confort,434,33,7.60,0.89,67073.27,0.42,0.64,0.85,False
22,home_construction,604,45,7.45,0.73,96920.36,0.61,-15.96,0.36,False
24,computers_accessories,7827,552,7.05,0.33,1059272.40,6.69,-2.10,0.00,True


In [17]:
# ============================================================
# FINAL CATEGORY PRIORITIZATION
# ============================================================

category_priority = category_opportunity.copy()

# ------------------------------------------------------------
# Priority logic
# ------------------------------------------------------------
# Tier 1 = strong repeat behavior + meaningful revenue
# Tier 2 = strong repeat behavior but weaker evidence/revenue
# Tier 3 = below repeat baseline
# ------------------------------------------------------------

category_priority["priority"] = "Low"

# Tier 1:
# Above repeat baseline + significant evidence + meaningful revenue
tier_1 = (
    (category_priority["repeat_item_share_pct"] > overall_repeat_baseline) &
    (category_priority["significant"] == True) &
    (category_priority["total_revenue"] >= 50000)
)

category_priority.loc[tier_1, "priority"] = "High"

# Tier 2:
# Above repeat baseline but doesn't satisfy all Tier 1 conditions
tier_2 = (
    (category_priority["repeat_item_share_pct"] > overall_repeat_baseline) &
    (~tier_1)
)

category_priority.loc[tier_2, "priority"] = "Medium"

# ------------------------------------------------------------
# Display final priority table
# ------------------------------------------------------------

priority_table = category_priority[
    [
        "product_category_name_english",
        "total_items",
        "repeat_items",
        "repeat_item_share_pct",
        "repeat_share_lift_pct",
        "total_revenue",
        "revenue_share_pct",
        "repeat_vs_onetime_pct",
        "adjusted_p_value",
        "significant",
        "priority"
    ]
].sort_values(
    ["priority", "total_revenue"],
    ascending=[True, False]
)

display(priority_table.round(2))

,product_category_name_english,total_items,repeat_items,repeat_item_share_pct,repeat_share_lift_pct,total_revenue,revenue_share_pct,repeat_vs_onetime_pct,adjusted_p_value,significant,priority
12,bed_bath_table,11115,1114,10.02,3.30,1241681.72,7.84,-4.89,0.00,True,High
24,computers_accessories,7827,552,7.05,0.33,1059272.40,6.69,-2.10,0.00,True,High
14,furniture_decor,8334,821,9.85,3.13,902511.79,5.70,-9.32,0.01,True,High
6,fashion_bags_accessories,2031,263,12.95,6.23,184273.54,1.16,-13.72,0.00,True,High
2,home_appliances,771,138,17.90,11.18,94990.43,0.60,-37.91,0.00,True,High
...,...,...,...,...,...,...,...,...,...,...,...
3,fashio_female_clothing,48,8,16.67,9.95,3425.39,0.02,NaN,NaN,NaN,Medium
4,la_cuisine,14,2,14.29,7.57,2388.54,0.02,NaN,NaN,NaN,Medium
1,arts_and_craftmanship,24,5,20.83,14.12,2184.14,0.01,NaN,NaN,NaN,Medium
0,diapers_and_hygiene,39,10,25.64,18.92,2141.27,0.01,-37.03,0.01,True,Medium


In [18]:
# ============================================================
# HIGH-PRIORITY CATEGORIES
# ============================================================

high_priority = priority_table[
    priority_table["priority"] == "High"
].copy()

print("HIGH-PRIORITY CATEGORIES")
print("=" * 80)

display(high_priority.round(2))

HIGH-PRIORITY CATEGORIES


,product_category_name_english,total_items,repeat_items,repeat_item_share_pct,repeat_share_lift_pct,total_revenue,revenue_share_pct,repeat_vs_onetime_pct,adjusted_p_value,significant,priority
12,bed_bath_table,11115,1114,10.02,3.30,1241681.72,7.84,-4.89,0.00,True,High
24,computers_accessories,7827,552,7.05,0.33,1059272.40,6.69,-2.10,0.00,True,High
14,furniture_decor,8334,821,9.85,3.13,902511.79,5.70,-9.32,0.01,True,High
6,fashion_bags_accessories,2031,263,12.95,6.23,184273.54,1.16,-13.72,0.00,True,High
2,home_appliances,771,138,17.90,11.18,94990.43,0.60,-37.91,0.00,True,High


In [19]:
# ============================================================
# PRIORITY SUMMARY
# ============================================================

priority_summary = (
    category_priority
    .groupby("priority")
    .agg(
        categories=("product_category_name_english", "count"),
        total_items=("total_items", "sum"),
        repeat_items=("repeat_items", "sum"),
        total_revenue=("total_revenue", "sum")
    )
    .reset_index()
)

priority_summary["repeat_item_share_pct"] = (
    priority_summary["repeat_items"]
    / priority_summary["total_items"]
) * 100

priority_summary["revenue_share_pct"] = (
    priority_summary["total_revenue"]
    / priority_summary["total_revenue"].sum()
) * 100

display(priority_summary.round(2))

,priority,categories,total_items,repeat_items,total_revenue,repeat_item_share_pct,revenue_share_pct
0,High,5,30078,2888,3482729.88,9.60,21.98
1,Low,47,69845,3618,10674886.46,5.18,67.38
2,Medium,20,12727,1062,1685936.90,8.34,10.64


In [20]:
# ============================================================
# BUSINESS CONCLUSION
# ============================================================

print("CATEGORY RETENTION CONCLUSION")
print("=" * 80)

print(f"Overall repeat-item baseline: {overall_repeat_baseline:.2f}%")
print(f"High-priority categories identified: {len(high_priority)}")

print("\nHigh-priority categories:")

for category in high_priority["product_category_name_english"]:
    print(f"- {category}")

print("\nInterpretation:")
print(
    "High-priority categories combine above-baseline repeat purchasing, "
    "statistical evidence of repeat behavior, and meaningful revenue contribution."
)

CATEGORY RETENTION CONCLUSION
Overall repeat-item baseline: 6.72%
High-priority categories identified: 5

High-priority categories:
- bed_bath_table
- computers_accessories
- furniture_decor
- fashion_bags_accessories
- home_appliances

Interpretation:
High-priority categories combine above-baseline repeat purchasing, statistical evidence of repeat behavior, and meaningful revenue contribution.
